# Notebook 1 — `text_conversion.ipynb`

**Sovereign Dialect-Bridge · Step 1 — Data Preparation & EDA**

Transformasi raw IndoSum (JSONL, nested token format) → Parquet bersih + EDA. Output yang dihasilkan dipakai oleh `training_sum.ipynb` (Notebook 3) dan `inference.ipynb` (Notebook 4).

**Input:** `dataset/indosum/{train,dev,test}.01.jsonl` (fold 1 dari 5-fold CV).
**Output:**
- `data/train.parquet`, `data/val.parquet`, `data/test.parquet` (test = 700 stratified samples)
- `data/train_sample.csv` (100 baris untuk inspeksi manual)

Notebook ini **tidak butuh GPU** — bisa dijalankan di CPU lokal.


## 1. Setup environment


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, re, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")


## 2. Install dependencies

Cukup dijalankan sekali per session.

```bash
pip install pandas pyarrow fastparquet matplotlib seaborn
```


## 3. Paths & configuration


In [ ]:
# Auto-detect project root: notebook bisa dijalankan dari notebook/ atau dari root
CWD = Path.cwd()
if (CWD / "dataset" / "indosum").exists():
    ROOT = CWD
elif (CWD.parent / "dataset" / "indosum").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError("Tidak menemukan dataset/indosum/ — jalankan dari root project atau notebook/")

DATASET_DIR = ROOT / "dataset" / "indosum"
OUTPUT_DIR  = ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Konfigurasi global (sinkron dengan CLAUDE.md)
TRAIN_SUBSET = 10000
N_VAL        = 750
N_TEST       = 700
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"ROOT       : {ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


## 4. Helper functions

Raw IndoSum punya struktur bersarang: `paragraphs[p][s][w]` (paragraf × kalimat × kata). Helper di bawah meratakan dan menggabung token sambil memperbaiki spasi sebelum tanda baca.


In [ ]:
def flatten_nested_tokens(nested) -> str:
    """Flatten arbitrary nested list of tokens into a clean string."""
    if isinstance(nested, str):
        return nested.strip()
    parts = []
    for item in nested:
        if isinstance(item, list):
            parts.append(flatten_nested_tokens(item))
        else:
            parts.append(str(item))
    text = " ".join(parts)
    text = re.sub(r"\s([.,!?;:)\]])", r"\1", text)   # rapikan spasi sebelum punctuation
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def sentence_to_str(sent_tokens) -> str:
    """One sentence (list of word tokens) → clean string."""
    return flatten_nested_tokens(sent_tokens)


def flatten_paragraphs_to_sentences(paragraphs):
    """3-level paragraphs → flat list of sentence strings."""
    sents = []
    for para in paragraphs:
        for sent in para:
            s = sentence_to_str(sent)
            if s:
                sents.append(s)
    return sents


def flatten_gold_labels(gold_labels):
    """2-level gold_labels (paragraph × sentence) → flat list of int."""
    out = []
    for para in gold_labels:
        for lab in para:
            out.append(int(bool(lab)))
    return out


def is_article_complete(text: str) -> bool:
    """Heuristik: artikel dianggap utuh jika diakhiri tanda baca akhir kalimat."""
    if not text:
        return False
    return text.rstrip()[-1:] in {".", "!", "?", '"', "'", ")"}


# Sanity check helper
_sample = [[["Halo", "dunia", "."], ["Selamat", "pagi", "!"]], [["Apa", "kabar", "?"]]]
print(flatten_paragraphs_to_sentences(_sample))
print(flatten_gold_labels([[True, False], [True]]))


## 5. Load JSONL records


In [ ]:
def load_jsonl(path: Path) -> list:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


raw_train = load_jsonl(DATASET_DIR / "train.01.jsonl")
raw_val   = load_jsonl(DATASET_DIR / "dev.01.jsonl")
raw_test  = load_jsonl(DATASET_DIR / "test.01.jsonl")

print(f"Raw train: {len(raw_train):,}")
print(f"Raw val  : {len(raw_val):,}")
print(f"Raw test : {len(raw_test):,}")
print("Sample keys:", list(raw_train[0].keys()))


## 6. Build typed records with full schema

Setiap record dibangun dengan kolom-kolom: `id, category, source, text, summary, sentences, sentence_labels, oracle_summary, word_count, summary_word_count, compression_ratio, n_sentences, is_complete`.

> **Catatan kritis:** `summary` di-flatten apa adanya — TIDAK di-lowercase, TIDAK di-stem. Ini penting agar ROUGE scoring tetap akurat.


In [ ]:
def build_record(raw: dict) -> dict | None:
    paragraphs = raw.get("paragraphs", [])
    summary    = raw.get("summary", [])
    gold       = raw.get("gold_labels", [])

    sentences        = flatten_paragraphs_to_sentences(paragraphs)
    sentence_labels  = flatten_gold_labels(gold)

    # Skip jika mismatch — akan dibuang juga oleh QC filter tapi hemat compute
    if len(sentences) != len(sentence_labels):
        return {
            "id": raw.get("id", ""),
            "category": raw.get("category", ""),
            "source": raw.get("source", ""),
            "text": flatten_nested_tokens(paragraphs),
            "summary": flatten_nested_tokens(summary),
            "sentences": sentences,
            "sentence_labels": sentence_labels,
            "oracle_summary": "",
            "word_count": 0,
            "summary_word_count": 0,
            "compression_ratio": 0.0,
            "n_sentences": len(sentences),
            "is_complete": False,
            "_mismatch": True,
        }

    text           = " ".join(sentences)
    summary_str    = flatten_nested_tokens(summary)
    oracle_summary = " ".join(s for s, lab in zip(sentences, sentence_labels) if lab == 1)

    w_count        = len(text.split())
    s_count        = len(summary_str.split())
    cr             = (s_count / w_count) if w_count > 0 else 0.0

    return {
        "id": raw.get("id", ""),
        "category": raw.get("category", ""),
        "source": raw.get("source", ""),
        "text": text,
        "summary": summary_str,
        "sentences": sentences,
        "sentence_labels": sentence_labels,
        "oracle_summary": oracle_summary,
        "word_count": w_count,
        "summary_word_count": s_count,
        "compression_ratio": round(cr, 4),
        "n_sentences": len(sentences),
        "is_complete": is_article_complete(text),
        "_mismatch": False,
    }


df_train_raw = pd.DataFrame([build_record(r) for r in raw_train])
df_val_raw   = pd.DataFrame([build_record(r) for r in raw_val])
df_test_raw  = pd.DataFrame([build_record(r) for r in raw_test])

print(f"Built train: {len(df_train_raw):,}")
print(f"Built val  : {len(df_val_raw):,}")
print(f"Built test : {len(df_test_raw):,}")
print("\nCategory distribution (train):")
print(df_train_raw['category'].value_counts())


## 7. Quality control filter

Buang record yang:
- `word_count < 50` (terlalu pendek)
- `summary_word_count < 10`
- `compression_ratio > 0.75` atau `< 0.03`
- `is_complete == False` (terpotong)
- `len(sentences) != len(sentence_labels)` (mismatch struktur)
- Tidak ada oracle sentence (`sum(sentence_labels) == 0`)


In [ ]:
def qc_filter(df: pd.DataFrame, name: str) -> pd.DataFrame:
    n0 = len(df)
    print(f"\n── QC filter on {name} ({n0:,} → ...) ──")

    mismatch = df["_mismatch"]
    print(f"  drop mismatch sentences/labels : {int(mismatch.sum()):,}")
    df = df[~mismatch].copy()

    too_short = df["word_count"] < 50
    print(f"  drop word_count < 50           : {int(too_short.sum()):,}")
    df = df[~too_short]

    short_sum = df["summary_word_count"] < 10
    print(f"  drop summary_word_count < 10   : {int(short_sum.sum()):,}")
    df = df[~short_sum]

    cr_high = df["compression_ratio"] > 0.75
    cr_low  = df["compression_ratio"] < 0.03
    print(f"  drop CR > 0.75                 : {int(cr_high.sum()):,}")
    print(f"  drop CR < 0.03                 : {int(cr_low.sum()):,}")
    df = df[~(cr_high | cr_low)]

    incomplete = ~df["is_complete"]
    print(f"  drop is_complete == False      : {int(incomplete.sum()):,}")
    df = df[df["is_complete"]]

    no_oracle = df["sentence_labels"].apply(lambda x: sum(x) == 0)
    print(f"  drop n_positive_labels == 0    : {int(no_oracle.sum()):,}")
    df = df[~no_oracle]

    df = df.drop(columns=["_mismatch"]).reset_index(drop=True)
    print(f"  → kept {len(df):,} / {n0:,} ({len(df)/n0:.1%})")
    return df


df_train = qc_filter(df_train_raw, "train")
df_val   = qc_filter(df_val_raw,   "val")
df_test  = qc_filter(df_test_raw,  "test")


## 8. Subsetting & stratified split

- Train: ambil **10.000 sampel** (cukup untuk hasil baik, hemat training time).
- Val: gunakan semua hasil filter dari `dev.01.jsonl`.
- Test: **700 sampel stratified per kategori** (representatif lintas 6 kategori berita).


In [ ]:
def stratified_subset(df: pd.DataFrame, n: int, by: str = "category", seed: int = RANDOM_SEED) -> pd.DataFrame:
    if len(df) <= n:
        return df.sample(frac=1, random_state=seed).reset_index(drop=True)
    # proporsional per category
    fracs = df[by].value_counts(normalize=True)
    per_cat = (fracs * n).round().astype(int).to_dict()
    parts = []
    for cat, k in per_cat.items():
        sub = df[df[by] == cat]
        k = min(k, len(sub))
        if k > 0:
            parts.append(sub.sample(k, random_state=seed))
    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    # patch jika kelebihan / kurang akibat rounding
    if len(out) > n:
        out = out.head(n)
    return out


df_train_final = stratified_subset(df_train, TRAIN_SUBSET)
df_val_final   = df_val.copy() if len(df_val) <= N_VAL else stratified_subset(df_val, N_VAL)
df_test_final  = stratified_subset(df_test, N_TEST)

print(f"Final train : {len(df_train_final):,}")
print(f"Final val   : {len(df_val_final):,}")
print(f"Final test  : {len(df_test_final):,}")
print("\nTest stratified per category:")
print(df_test_final["category"].value_counts())


## 9. Save Parquet & sample CSV


In [ ]:
def save_split(df: pd.DataFrame, path: Path):
    df.to_parquet(path, index=False)
    print(f"  saved {path.name:20s}  rows={len(df):,}  cols={len(df.columns)}")


save_split(df_train_final, OUTPUT_DIR / "train.parquet")
save_split(df_val_final,   OUTPUT_DIR / "val.parquet")
save_split(df_test_final,  OUTPUT_DIR / "test.parquet")

# Sample CSV (drop kolom list untuk readability)
csv_cols = ["id", "category", "source", "text", "summary",
            "word_count", "summary_word_count", "compression_ratio", "n_sentences"]
df_train_final[csv_cols].head(100).to_csv(OUTPUT_DIR / "train_sample.csv", index=False)
print(f"  saved train_sample.csv  rows=100")


## 10. EDA — Distribusi panjang artikel & ringkasan

Target compression ratio natural IndoSum: **~0.225**. Kalau distribusi train hasil filter masih di sekitar nilai itu, berarti QC tidak menyebabkan bias.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

stats = df_train_final[["word_count", "summary_word_count", "compression_ratio"]].describe(
    percentiles=[0.05, 0.5, 0.95]
).round(2)
print("Train statistics:")
print(stats)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df_train_final["word_count"], bins=50, color="#3b82f6", alpha=0.85)
axes[0].set_title("Article word_count"); axes[0].set_xlabel("words")
axes[1].hist(df_train_final["summary_word_count"], bins=50, color="#10b981", alpha=0.85)
axes[1].set_title("Summary word_count"); axes[1].set_xlabel("words")
axes[2].hist(df_train_final["compression_ratio"], bins=50, color="#f59e0b", alpha=0.85)
axes[2].axvline(0.225, color="red", linestyle="--", label="natural CR ≈ 0.225")
axes[2].legend(); axes[2].set_title("Compression ratio"); axes[2].set_xlabel("summary/article")
plt.tight_layout(); plt.show()


## 11. EDA — Distribusi kategori


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, df, title in zip(axes, [df_train_final, df_val_final, df_test_final], ["train", "val", "test"]):
    vc = df["category"].value_counts()
    ax.barh(vc.index[::-1], vc.values[::-1], color="#6366f1")
    ax.set_title(f"{title}  (n={len(df):,})")
    ax.set_xlabel("count")
plt.tight_layout(); plt.show()

# Cek imbalance
print("\nImbalance ratio (max/min per split):")
for df, name in [(df_train_final, "train"), (df_val_final, "val"), (df_test_final, "test")]:
    vc = df["category"].value_counts()
    print(f"  {name}: {vc.max() / max(vc.min(), 1):.2f}x")


## 12. EDA — Korelasi word_count vs summary_word_count


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df_train_final, x="word_count", y="summary_word_count",
                hue="category", alpha=0.5, s=15, ax=ax)
ax.set_title("Article length vs summary length")
ax.set_xlabel("word_count"); ax.set_ylabel("summary_word_count")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

corr = df_train_final[["word_count", "summary_word_count"]].corr().iloc[0, 1]
print(f"Pearson correlation: {corr:.3f}")


## 13. EDA — Top words artikel vs summary


In [ ]:
ID_STOPWORDS = {
    "yang","di","ke","dari","dan","atau","ini","itu","dalam","untuk","pada","dengan",
    "sebagai","oleh","akan","juga","sudah","tidak","bukan","ada","tak","tapi","saat",
    "telah","saya","kami","kita","mereka","dia","ia","nya","ku","mu","se","dapat","bisa",
    "para","adalah","yaitu","yakni","hingga","sampai","jika","kalau","karena","sebab",
    "agar","supaya","bahwa","tentang","secara","seperti","menjadi","lebih","sangat",
    "masih","baru","banyak","semua","tahun","hari","orang"
}

def top_tokens(texts, k=15, min_len=3):
    tokens = []
    for t in texts:
        for w in re.findall(r"[A-Za-zÀ-ÿ]+", t.lower()):
            if len(w) >= min_len and w not in ID_STOPWORDS:
                tokens.append(w)
    return Counter(tokens).most_common(k)


print("Top-15 di TEXT:")
for w, c in top_tokens(df_train_final["text"], 15):
    print(f"  {w:20s}  {c:,}")
print("\nTop-15 di SUMMARY:")
for w, c in top_tokens(df_train_final["summary"], 15):
    print(f"  {w:20s}  {c:,}")


## ✅ Selesai

Output:
- `data/train.parquet` — input training untuk Notebook 3 (`training_sum.ipynb`)
- `data/val.parquet`   — validation set
- `data/test.parquet`  — 700 stratified samples untuk eval di Notebook 4
- `data/train_sample.csv` — 100 baris untuk inspeksi manual

**Langkah selanjutnya:** jalankan `training_normalizer.ipynb` untuk training Stage 1 (mT5-small dialect → BI baku).
